# Integrated night-security simulation

Run cells in order. No hardware, microphone permission or GPU is required. The first run downloads project dependencies and the declared ESC-50 subset. This notebook has been syntax-checked; execution on Google Colab has not yet been verified. The underlying Python simulation has been run separately.

All channels here are simulated. Dataset audio is public proxy data, not a recording of the intended room. MATLAB and Arduino target compilation are separate checks; this notebook does not claim to run them.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil
PROJECT = Path.cwd() / "complex-embedded-iot-based-security-system"
if not PROJECT.exists():
    subprocess.run(["git", "clone", "https://github.com/konark-icdesign/complex-embedded-iot-based-security-system.git", str(PROJECT)], check=True)
print("Project revision:")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT, check=True)
def run(*args, cwd=PROJECT):
    subprocess.run(list(args), cwd=cwd, check=True)
run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")


## Check the individual modules
Python regressions cover invalid signals, timestamp handling, camera lighting, sensor filtering, fusion and durable mock alert retries. The focused audio experiment uses the same DSP implementation as the full system.


In [ ]:
run(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-v")
run(sys.executable, "run_audio.py", cwd=PROJECT / "experiments/audio_room")
run(sys.executable, "-m", "unittest", "test_audio", "-v", cwd=PROJECT / "experiments/audio_room")
print((PROJECT / "experiments/audio_room/results/summary.json").read_text())


## Run all modules together
Use 10 seeds for the published 520-trial experiment; use 1 for a quicker 52-scenario check. Allow several minutes. Download failures must be resolved before treating the real-audio challenge as executed.


In [ ]:
SEEDS = 10
run(sys.executable, "scripts/fetch_real_audio.py")
run(sys.executable, "run_simulation.py", "--seeds", str(SEEDS))
summary = json.loads((PROJECT / "results/summary.json").read_text())
print(json.dumps(summary, indent=2))


In [ ]:
from IPython.display import display, Image
for filename in ["scenario_outcomes.png", "camera_blocked_timeline.png", "lighting_test.png", "ultrasonic_test.png"]:
    display(Image(filename=str(PROJECT / "results" / filename)))


## Check Arduino logic on the host
This runs C++ logic tests and replay when GCC is available. It does not connect to a board or compile the Arduino target. Sanitizer failures are errors to inspect, not detection results.


In [ ]:
if shutil.which("g++") and shutil.which("make"):
    run("make", "embedded")
    run(sys.executable, "scripts/verify_embedded_replay.py")
else:
    print("C++ host checks NOT RUN: g++ or make is unavailable.")


## Interpret the results
Read `docs/experiment_report.md` alongside the new results. Known intrusion misses and false alerts are part of the measured outcome. A successful test suite means regression checks passed; it does not mean every intrusion was detected. No real alert is sent.
